# Phase 1: Refine stage

Codes and justifications are refined using the feebback generated in the previous stage (via rule based function). This process produces final codes and justifications

In [ ]:
import outlines
import transformers
import torch
import os
import torch
from pydantic import BaseModel, Field
from typing import List

from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


### Read Data

In [ ]:
import pandas as pd
import json

df = pd.read_parquet(FEEDBACK_DIR / "current_feedback_final_TEST.parquet")

df.columns



Index(['unit_id', 'int_id', 'interview_excerpt', 'segment_id', 'codes',
       'justifications', 'status', 'error', 'code_id', 'code_uid',
       'code_label', 'code_label_justification', 'justification_label',
       'justification_eval_explanation'],
      dtype='object')

### Load LLM model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id) #Mistral's tokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto" #A100 or H100
)

print("Mistral-7B-Instruct-v0.3 is loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Mistral-7B-Instruct-v0.3 is loaded.


### LLM configurations w. Outlines

In [ ]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig.from_model_config(model.config)

model.generation_config.max_new_tokens = 600 #max token output
#model.generation_config.min_new_tokens = 40 #min new tokens
model.generation_config.temperature = 0.0 # Temp set to 0.0 because it is overridden by Outlines in structured output
model.generation_config.do_sample = False #LLM chooses most likely next token = False
model.generation_config.pad_token_id = tokenizer.eos_token_id # Mistral does not have a pad token, so we pad with EOS
model.generation_config.eos_token_id = tokenizer.eos_token_id #Mistral's end-of-sequence token

### Import Outlines for validation of output

In [ ]:
import outlines

outlines_model = outlines.from_transformers(model, tokenizer)

### Analysis prompt - refine stage

### Format revise stage prompt in INST (chat_template)

In [ ]:
def format_prompt(content, tokenizer): #Taking analysis prompt and sending it to model in instrct mode
    messages = [
        {"role": "user", "content": content}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

### Code refine decision tree function

In [ ]:
def get_refine_action(row):
    # Special case: not_relevant (no justification should exist)
    if row['code_label'] == 'not_relevant':
        return 'keep_not_relevant'

    # If code is good
    if row['code_label'] == 'good':
        if row['justification_label'] == 'good':
            return 'keep'
        else:
            return 'fix_justification'

    # If code is bad --> always refine both (justifications follow codes)
    else:
        return 'fix_both'

In [ ]:
# Apply to DF as new column
df["refine_action"] = df.apply(get_refine_action, axis=1)

In [ ]:
df["refine_action"].value_counts()

refine_action
keep                 202
fix_both             201
keep_not_relevant    178
fix_justification     69
Name: count, dtype: int64

In [ ]:
df.columns

Index(['unit_id', 'int_id', 'interview_excerpt', 'segment_id', 'codes',
       'justifications', 'status', 'error', 'code_id', 'code_uid',
       'code_label', 'code_label_justification', 'justification_label',
       'justification_eval_explanation', 'refine_action'],
      dtype='object')

### 2 x Analysis promps that matches the outcome space for decisions (fix_justification, fix_both)

In [ ]:
#fix_justification analysis prompt for fix_justification action
def get_fix_justification_prompt(interview_excerpt, justification_eval_explanation, code, justification):

    content = f"""
    ### TASK
    You are an expert qualitative researcher conducting Thematic Analysis (Braun & Clarke 2006).
    Your task is to refine the justification ONLY.

    ### CONTEXT
    The research objective is to understand the interviewees’ individual experiences of the profound transformations in Danish society
    and family life from the 1960s-1980s, characterized by changing gender roles, women’s increasing labour market participation,
    the weakening of traditional marriage, and the expansion of family policies.

    ### INTERVIEW EXCERPT
    {interview_excerpt}

    ### CURRENT OUTPUT
    Code: {code}
    Justification: {justification}

    ### EVALUATION CONTEXT
    {justification_eval_explanation}

    ### INSTRUCTIONS
    - The code is fixed and must not be changed
    - Rewrite the justification only so it clearly explains why the revised code applies in 1-2 short sentences
    - Improve analytical clarity and specificity
    - Do NOT paraphrase the interview excerpt

    ### GUIDELINES
    - Justification must explain the reasoning explicitly

    ### IMPORTANT
    - Do not include any additional text
    """

    return content

In [ ]:
#fix_both analysis prompt for fix_both action

def get_fix_both_prompt(interview_excerpt,code_label_justification, justification_eval_explanation, code, justification):

    content = f"""
    ### TASK
    You are an expert qualitative researcher conducting Thematic Analysis (Braun & Clarke 2006).
    Your task is to refine an existing code and its justification.

    ### CONTEXT
    The research objective is to understand the interviewees’ individual experiences of the profound transformations in Danish society
    and family life from the 1960s-1980s, characterized by changing gender roles, women’s increasing labour market participation,
    the weakening of traditional marriage, and the expansion of family policies.

    ### INTERVIEW EXCERPT
    {interview_excerpt}

    ### CURRENT OUTPUT (TO BE REFINED)
    Code: {code}
    Justification: {justification}

    ### EVALUATION FEEDBACK
    Code issue:
    {code_label_justification}

    Justification issue:
    {justification_eval_explanation}

    ### INSTRUCTIONS

    Improve the existing code and justification based on the feedback:

    - Revise the code so it accurately reflects the meaning of the interview excerpt
    - Each code must have a clear, concise name in one shorthand phrase.
    - Keep the revised code close to the original where possible, but correct the identified issues
    - Ensure the code is concise and analytically precise (not vague or overly general)

    - Rewrite the justification so it clearly explains why the revised code applies in 1-2 short sentences
    - Explicitly link elements of the excerpt to the code
    - Address the issues identified in the feedback
    - Do NOT paraphrase the interview excerpt

    ### GUIDELINES
    - Focus on improving the existing analysis, not generating a completely new one
    - Maintain alignment between code and justification
    - Base the analysis primarily on the informant’s responses (A:)

    ### IMPORTANT
    - Do not introduce new themes unrelated to the original code
    - Do not ignore the EVALUATION FEEDBACK
    - Do not include any additional text
    """

    return content

### Defining 2 x output schema with Pydantic


In [ ]:
# Schema Output for fix_justification
from pydantic import BaseModel, Field, field_validator

class FixJustificationResponse(BaseModel):
    justification: str = Field(min_length=1)

    @field_validator("justification")
    @classmethod
    def no_empty_or_whitespace(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("Justification cannot be empty")
        return v

In [ ]:
#SChema Output for fix_both

class FixBothResponse(BaseModel):
    code: str = Field(min_length=1)
    justification: str = Field(min_length=1)

    @field_validator("code", "justification")
    @classmethod
    def no_empty_or_whitespace(cls, v: str) -> str:
        v = v.strip()
        if not v:
            raise ValueError("Field cannot be empty")
        return v

### Function for reg.ex normalization of codes - and ONLY codes

In [ ]:
import re

def normalize_code_regex_only(code: str) -> str:
    if code is None:
        return None

    code = code.strip()


    # Transform camelCase to underscores
    code = re.sub(r'(?<!^)(?=[A-Z])', '_', code)

    # lowercase
    code = code.lower()

    #Know error for misspelling of relevant
    code = code.replace("relevent", "relevant")

    # replace whitespace and hyphens with underscore
    code = re.sub(r"[\s\-]+", "_", code)

    # remove all other tokens than letters, numbers and underscore
    code = re.sub(r"[^a-z0-9_]", "", code)

    # Collapes multiple underscores
    code = re.sub(r"_+", "_", code)

    # remove underscores in start and end of code
    code = code.strip("_")

    return code

In [ ]:
import json
import time

def refine_with_retry(prompt_content, response_model, tokenizer, max_retries=3, retry_delay=1):

    prompt_inst = format_prompt(prompt_content, tokenizer)

    for attempt in range(max_retries):
        try:
            # Outlines usually returns the validated object directly if passed a schema
            result = outlines_model(prompt_inst, response_model)

            # If outlines_model returns a string, keep these lines;
            # if it returns the object directly, just return result.
            parsed = json.loads(result) if isinstance(result, str) else result
            return response_model(**parsed) if isinstance(parsed, dict) else parsed

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
                continue
            raise ValueError(f"Failed after {max_retries} attempts: {e}")

### Run pipeline - call functions

In [ ]:
import time
import pandas as pd

unique_int_ids = df["int_id"].unique()
total_interviews = len(unique_int_ids)

print(f"Starting thematic coding refine phase... ({total_interviews} interviews)\n")

start_total = time.time()
results = []

for i, int_id in enumerate(unique_int_ids, start=1):
    interview_start = time.time()

    print(f"\n--- Interview {i}/{total_interviews}: {int_id} ---")

    df_subset = df[df["int_id"] == int_id]

    print(f"Number of excerpts: {len(df_subset)}")
    print(df_subset["refine_action"].value_counts())

    for _, row in df_subset.iterrows():
        print(f"Processing row {row['code_uid']} with action: {row['refine_action']}") # DEBUG

        try:
            match row["refine_action"]:

                case "fix_both":
                    ##skal man her iterere over listen af koder/justifications eller er det allerede kun et udfald pr linje?
                    content = get_fix_both_prompt(row["interview_excerpt"], row["code_label_justification"], row["justification_eval_explanation"], row["codes"], row["justifications"])
                    validated = refine_with_retry(content, FixBothResponse, tokenizer)
                    print(f"pydantic output: {validated}")
                    codes = normalize_code_regex_only(validated.code)
                    justifications =validated.justification.strip()

                case "fix_justification":
                    content = get_fix_justification_prompt(row["interview_excerpt"], row["justification_eval_explanation"], row["codes"], row["justifications"])
                    validated = refine_with_retry(content, FixJustificationResponse, tokenizer)
                    print(f"pydantic output: {validated}")
                    codes = row["codes"]
                    justifications = validated.justification.strip()

                case "keep":
                    codes = row["codes"]
                    justifications = row["justifications"]

                case "keep_not_relevant":
                    codes = "not relevant"
                    justifications = "not relevant"


            #If no error write none in refine_error col
            row_dict = {
            "refine_error": None,
            "codes_final": codes,
            "justifications_final": justifications,
            "code_uid": row["code_uid"],
            "refine_action": row["refine_action"]
            }

        except Exception as e:
            print(f"DEBUG: Row {row['code_uid']} failed with error: {type(e).__name__} - {e}")
            row_dict = {
            "refine_error": str(e),
            "codes_final": None,
            "justifications_final": None,
            "code_uid": row["code_uid"],
            "refine_action": row["refine_action"]
            }

        results.append(row_dict)

    interview_time = time.time() - interview_start
    print(f"Finished interview {int_id} in {round(interview_time, 2)} sec")

total_time = time.time() - start_total

results_df = pd.DataFrame(results)
print(results_df.head(10))
print(f"\nTotal time: {round(total_time, 2)} sec ({round(total_time/60, 2)} min)")

### Saving results

In [ ]:
import datetime
import shutil


# merge results on copy of original df
df_results_refine_final = df.copy().merge(
    results_df ,
    on="code_uid",
    how="left"
)

In [ ]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")


# Save as PARQUET (source of truth for the next step of self_refine flow)
timestamp_path_parquet = FINAL_DIR / f"results_refine_final{timestamp}.parquet"
current_path_parquet = FINAL_DIR / "currentTEST.parquet"

df_results_refine_final.to_parquet(timestamp_path_parquet, index=False)
shutil.copy(timestamp_path_parquet, current_path_parquet)

